In [1]:
!pip install langgraph-checkpoint-sqlite
!pip install \
  langchain \
  langchain-core \
  langchain-community \
  langchain-anthropic \
  langchain-chroma \
  chromadb
!pip install a2a-sdk

In [9]:
from typing import TypedDict
from typing import Any
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma(embedding_function=embeddings)
REVIEW_THRESHOLD = 0.70
MAX_REVISIONS = 3

class AgentState(TypedDict):
    # ── inputs ──
    repo_path: str                          # local path or remote URL


    repo_summary: str                       # Legacy code structure
    prior_architectures: list[str]          # Architectures generated so far
    review_history: list[dict]              # Review comments

    # ── working outputs ──
    architecture_doc: str
    confidence_score: float
    review_critique: str
    revision_count: int

    # ── MCP tool outputs ──
    cost_estimate: dict[str, Any]


    # ── pipeline control ──
    guardrail_error: str | None             # Gaurdrails
    human_approved: bool                    # set by HITL node

    # ── message trace (for observability) ──
    messages: Annotated[list, add_messages]
    chunks: list
    cve_results: str


def initial_state(code: str) -> AgentState:
    return AgentState(

        repo_summary=code,
        prior_architectures=[],
        review_history=[],
        architecture_doc="",
        confidence_score=0.0,
        review_critique="",
        revision_count=0,
        cost_estimate={},
        guardrail_error=None,
        human_approved=False,
        messages=[],
    )



/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/tmp/ipykernel_29996/39506942.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a tok

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_29996/39506942.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(embedding_function=embeddings)


In [6]:
from a2a.server.agent_execution import AgentExecutor
import requests
from typing import Dict,List,Any
class NvdCveAgentExecutor(AgentExecutor):

    async def execute(
        self,
        input_data: str,
        context: Dict[str, Any] | None = None,
    ) -> Dict[str, Any]:



        url = "https://services.nvd.nist.gov/rest/json/cves/2.0"

        response = requests.get(
            url,
            params={"keywordSearch": input_data},
            timeout=30,
        )

        response.raise_for_status()

        data = response.json()

        vulnerabilities = data.get("vulnerabilities", [])

        processed = []

        for item in vulnerabilities[:10]:

            cve = item.get("cve", {})

            cve_id = cve.get("id", "UNKNOWN")

            descriptions = cve.get("descriptions", [])

            description = ""

            if descriptions:
                description = descriptions[0].get("value", "")

            processed.append({
                "cve_id": cve_id,
                "description": description
            })

        return {
            "keyword": keyword,
            "cves": processed
        }


In [3]:
# Legacy code ingestor
from google.colab import drive
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
def chunking():
  drive.mount('/content/drive')
  file_path = '/content/drive/MyDrive/combined_output.txt'
  with open(file_path, 'r', encoding='utf-8') as f:
      content = f.read()
  content


  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=500,
      chunk_overlap=50)
  chunks = text_splitter.split_text(content)

In [ ]:
# Node to generate architecute
'''
LLM : Claude OPS
System Instructions: Agent.md
Temperature: 0.3

'''
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
import os
def node_architecture(state: AgentState) -> dict:
   agent_path = '/content/drive/MyDrive/agent.md'
   with open(agent_path, 'r', encoding='utf-8') as f:
      agent_instructions = f.read()
   api_key = userdata.get('OPENAI_API_KEY')
   os.environ["ANTHROPIC_API_KEY"] = api_key
   llm = ChatAnthropic(
    model="claude-opus-4-20250514",
    temperature=0.3)
   tools=[]
   prior = "\n\n".join(state["prior_architectures"][-2:]) or "None available."
   critique = state["review_critique"] or "First attempt — no prior critique."

   prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=agent_instructions),
        HumanMessage(content=f"""
        Codebase summary:
        {state['repo_summary']}

        Prior architecture examples for reference:
        {prior}

        Reviewer critique from last attempt (address these points):
        {critique}

        Revision number: {state['revision_count']}

        Generate the architecture document now.
        """),    ])
   llm = create_react_agent(
      model=llm,
      tools=tools,
      prompt=prompt)
   response = llm.invoke(prompt.format_messages())
   doc = response.content
   # there is a scope to improve the guardrail logic in the future to capture errors from llm invoke.
   return {
        "architecture_doc": doc,
        "guardrail_error": None,
        "messages": [response],
    }



In [4]:
def guardrail_review_score(score: float) -> str | None:
    if not (0.0 <= score <= 1.0):
        return f"Review guardrail: invalid score {score}"
    return None

In [ ]:
import json
REVIEW_SYSTEM = """You are an independent architecture reviewer. Evaluate the provided architecture
document and return a JSON object with exactly these fields:

{
  "confidence_score": <float 0.0-1.0>,
  "strengths": [<string>, ...],
  "weaknesses": [<string>, ...],
  "critique": "<actionable feedback for the architecture agent>"
}

Scoring guide:
  0.9-1.0  production-ready, minimal changes needed
  0.7-0.89 good, minor improvements needed
  0.5-0.69 significant gaps, revise before proceeding
  0.0-0.49 fundamental issues, major rework needed

Return ONLY valid JSON — no markdown fences, no preamble."""

def node_review_agent(state: AgentState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=REVIEW_SYSTEM),
        HumanMessage(content=f"""
           Architecture document to review:
           {state['architecture_doc']}
           Codebase context:{state['repo_summary']}"""),])
    '''
    LLM : Claude OPS
    Temperature: 0.7
    '''
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ["ANTHROPIC_API_KEY"] = api_key
    llm = ChatAnthropic(
    model="claude-opus-4-20250514",
    temperature=0.7)
    response = llm.invoke(prompt.format_messages())
    try:
        result = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback: extract JSON from response
        match = re.search(r"\{.*\}", response.content, re.DOTALL)
        result = json.loads(match.group()) if match else {"confidence_score": 0.0, "critique": "Parse error"}

    score = float(result.get("confidence_score", 0.0))
    score_error = guardrail_review_score(score)
    if score_error:
        return {"guardrail_error": score_error}

    review_entry = {
        "revision": state["revision_count"],
        "score": score,
        "critique": result.get("critique", ""),
    }

    return {
        "confidence_score": score,
        "review_critique": result.get("critique", ""),
        "review_history": state["review_history"] + [review_entry],
        "guardrail_error": None,
        "messages": [response],
    }




In [ ]:
from typing import Annotated, Any, Literal
def route_after_review(state: AgentState) -> Literal["architecture", "leader", "error"]:
    if state["guardrail_error"]:
        return "error"
    if state["confidence_score"] < REVIEW_THRESHOLD:
        if state["revision_count"] >= MAX_REVISIONS:
            # Too many revisions — surface for human decision
            return "leader"
        return "architecture"
    return "leader"

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
def node_leader(state: AgentState) -> dict:

    final_architecture=state['architecture_doc']
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200)
    docs = text_splitter.create_documents([final_architecture])
    state['chunks']=docs
    vector_db.add_documents(docs)






In [10]:
from mcp import ClientSession
from langchain_core.runnables import RunnableLambda

async def node_estimate_cost(state: AgentState) -> dict:
  session = ClientSession("http://localhost:8000")
  await session.connect()
  retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3})
  docs = retriever.invoke("Terraform code")
  async with session:
     prompt = await session.get_prompt("ask_about_topic", {"topic": "Model Context Protocol"})
  llm = ChatAnthropic(
    model="claude-opus-4-20250514",
    temperature=0.3)
  response = llm.invoke(prompt.format_messages())
  doc = response.content
  return {
        "cost_estimate": doc,
        "guardrail_error": None,
        "messages": [response],
    }





In [ ]:
from keybert import KeyBERT
from langchain_text_splitters import RecursiveCharacterTextSplitter
async def node_search_cve(state: AgentState) -> dict:

    executor = NvdCveAgentExecutor()
    kw_model = KeyBERT()

    all_keywords=set()
    txt=" ".join(state.get("chunks"))
    keywords = kw_model.extract_keywords(
        txt,
        keyphrase_ngram_range=(1, 3),
        stop_words='english',
        top_n=10
    )
    for keys in enumerate(keywords):
        all_keywords.add(keys[1][0])
    technology=" ".join(all_keywords)

    result = await executor.execute({
        "keyword": technology
    })

    return {
        "cve_results": result["cves"]
    }

In [ ]:
def node_human_review(state: AgentState) -> dict:

    if not state.get("human_approved"):
        # State will be persisted; graph waits for external resume
        return {}
    return {"human_approved": True}

In [ ]:
def node_aggregator(state: AgentState) -> dict:
    """Merge all outputs into a final delivery package."""
    final_output = {
        "architecture": state["architecture_doc"],
        "confidence_score": state["confidence_score"],
        "cost_estimate": state["cost_estimate"],
        "review_history": state["review_history"],
        "revision_count": state["revision_count"],
    }

    # Final compliance guardrail
    if not state["architecture_doc"]:
        return {"guardrail_error": "Final guardrail: empty architecture doc"}

    summary_msg = HumanMessage(
        content=f"[PIPELINE COMPLETE] Score: {state['confidence_score']:.2f} | "
                f"Cost: ${state['cost_estimate'].get('monthly_total_usd', 0):,.0f}/mo | "
                f"Epics: {len(state['jira_epics'])}"
    )
    return {"messages": [summary_msg]}

In [ ]:
def route_after_human(state: AgentState) -> Literal["aggregator", "architecture"]:
    """Human can approve or send back for revision."""
    return "aggregator" if state["human_approved"] else "architecture"



In [ ]:
    def load_memory(key: str, default=None):
        cursor = conn.execute(
            "SELECT value FROM memory_store WHERE key = ?",
            (key,)
        )

        row = cursor.fetchone()

        if row is None:
            return default

        return json.loads(row[0])
    def load_memory_into_state(state: AgentState) -> AgentState:
        """
        Load persistent memory from checkpoints.db
        """

        state["prior_architectures"] = load_memory(
            "architectures",
            []
        )

        state["review_history"] = load_memory(
            "review_history",
            []
        )

        return state

In [ ]:
def save_memory(key: str, value):
    conn.execute(
        """
        INSERT OR REPLACE INTO memory_store (key, value)
        VALUES (?, ?)
        """,
        (key, json.dumps(value))
    )

    conn.commit()
def save_memory_from_state(state: AgentState) -> None:
    """
    Persist outputs into SQLite-backed memory store.
    """

    # -----------------------------
    # Save architecture history
    # -----------------------------
    architecture_doc = state.get("architecture_doc")

    if architecture_doc:

        architectures = load_memory(
            "architectures",
            []
        )

        architectures.append(architecture_doc)

        # keep last 5
        architectures = architectures[-5:]

        save_memory(
            "architectures",
            architectures
        )

    # -----------------------------
    # Save review history
    # -----------------------------
    review_history = state.get("review_history")

    if review_history:

        existing_reviews = load_memory(
            "review_history",
            []
        )

        updated_reviews = (
            existing_reviews + review_history
        )[-20:]  # keep last 20

        save_memory(
            "review_history",
            updated_reviews
        )

In [2]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
# Context manager to initiate resources
def pipeline(code: str, thread_id: str = "default") -> dict:
  conn = sqlite3.connect("checkpoints.db")

  conn.execute("""
    CREATE TABLE IF NOT EXISTS memory_store (
        key TEXT PRIMARY KEY,
        value TEXT
    )
    """)

  conn.commit()
  with SqliteSaver.from_conn_string("checkpoints.db") as checkpointer:
      graph = orchestrator(checkpointer=checkpointer)
      config = {"configurable": {"thread_id": thread_id}}
      state = initial_state(code)
      state = load_memory_into_state(state)
      for event in graph.stream(state, config=config, stream_mode="updates"):
          for node_name, node_output in event.items():
              msgs = node_output.get("messages", [])
              score = node_output.get("confidence_score")
              error = node_output.get("guardrail_error")

              print(f"[{node_name}]", end=" ")
              if error:
                 print(f"BLOCKED: {error}")
              elif score is not None:
                 print(f"confidence={score:.2f}")
              elif msgs:
                 preview = msgs[-1].content[:80].replace("\n", " ")
                 print(f"{preview}...")
              else:
                 print("ok")

                # Retrieve final state
                final = graph.get_state(config)
                final_values = final.values

                # ── Simulate human approval (in production: wait for UI callback) ──
                if final.next and "human_review" in final.next:
                  print("\n[HITL] Pipeline paused for human review.")
                  print(f"  Architecture score: {final_values['confidence_score']:.2f}")
                  print(f"  Epics ready: {len(final_values['cve'])}")
                  print("  [AUTO-APPROVING for demo purposes]")

                  for event in graph.stream(
                      {"human_approved": True},
                        config=config,
                        stream_mode="updates",
                      ):
                          for node_name, node_output in event.items():
                              print(f"[{node_name}] resumed")

                  final = graph.get_state(config)
                  save_memory_from_state(final.values)

                  return final.values








/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
def orchestrator(checkpointer=None):
    g = StateGraph(AgentState)
    # nodes
    g.add_node("architecture",     node_architecture)
    g.add_node("review",            node_review_agent)
    g.add_node("leader",  node_leader)

    g.add_node("estimate_cost",node_estimate_cost)
    g.add_node("search_cve",node_search_cve)
    # Entry point
    g.add_edge(START, "architecture")
    g.add_edge("architecture", "review")
    g.add_conditional_edges("review", route_after_review, {
        "architecture":    "architecture",   # revise loop
        "leader": "leader",
        "error":           "error",
    })
    g.add_edge("leader","estimate_cost")
    g.add_edge("leader","search_cve")
    g.add_node("human_review",      node_human_review)
    g.add_node("aggregator",        node_aggregator)
    g.add_conditional_edges("human_review", route_after_human,
                            {"aggregator": "aggregator", "architecture": "architecture"})
    g.add_edge("aggregator", END)
    g.add_edge("error",      END)
    return g.compile(
        checkpointer=checkpointer,
        interrupt_before=["human_review"],   # HITL pause
    )

In [ ]:
import sys
if __name__ == "__main__":

      result = pipeline(code=chunking(), thread_id="run-001")
      print("\n" + "="*60)
      print("PIPELINE COMPLETE")
      print("="*60)
      print(f"Confidence score : {result.get('confidence_score', 0):.2f}")
      print(f"Revisions        : {result.get('revision_count', 0)}")
      print(f"Monthly cost est : ${result.get('cost_estimate', {}).get('monthly_total_usd', 0):,.0f}")
      print(f"Jira epics       : {len(result.get('jira_epics', []))}")
      if result.get("guardrail_error"):
        print(f"\nHalted by guardrail: {result['guardrail_error']}")
      else:
        print("\nArchitecture document preview:")
        print(result.get("architecture_doc", "")[:500] + "...")
